# Broad cache (b1)

Builds the b1 cache from the released training recordings for subjects 1 to 32 and holdout clips for subjects 1 to 39. It uses the same filtering and resampling as d2 and saves T5-large layer-12 word embeddings. Each subject's holdout clips are scaled using their own per-channel IQR.

In [ ]:
from pathlib import Path

OUTPUT_DIR = Path(".")
SCRATCH = (Path("/kaggle/temp") if Path("/kaggle/temp").exists() else Path("/tmp")) / "pnpl_b1"
DOWNLOAD_DIR = SCRATCH / "data"
HOLDOUT_DIR = SCRATCH / "holdout"
CACHE_DIR = OUTPUT_DIR / "cache"
T5_CACHE_DIR = None
CACHE_VERSION = "b1"
FREQ = 50.0
LOWPASS = 40.0
T5_MODEL = "t5-large"
T5_LAYER = 12
LABELLED = list(range(1, 33))
MIN_FREE_GB = 3.0

In [ ]:
import subprocess
import sys

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "pnpl==0.2.0",
        "mne",
        "h5py",
        "transformers",
    ],
    check=True,
)

In [ ]:
import os
import re
import shutil
import time
import h5py
import mne
import numpy as np
import pandas as pd
from huggingface_hub import HfApi, hf_hub_download
from pnpl.competition import LibriBrainCompetitionHoldout, load_vocabulary

if T5_CACHE_DIR is not None:
    T5_CACHE_DIR = Path(T5_CACHE_DIR)
    for filename in (f"meta_{CACHE_VERSION}.npz", f"t5_{CACHE_VERSION}.bin"):
        path = T5_CACHE_DIR / filename
        if not path.is_file():
            raise FileNotFoundError(f"Missing T5 input: {path}")

mne.set_log_level("ERROR")
for path in (DOWNLOAD_DIR, HOLDOUT_DIR, CACHE_DIR):
    path.mkdir(parents=True, exist_ok=True)
try:
    from kaggle_secrets import UserSecretsClient

    os.environ.setdefault("HF_TOKEN", UserSecretsClient().get_secret("HF_TOKEN"))
except Exception:
    pass
TOKEN = os.environ.get("HF_TOKEN")
print("HF_TOKEN", "found" if TOKEN else "missing (anonymous downloads, slower)")
vocabulary = load_vocabulary("primary")


def normalise_word(w):
    w = str(w).replace("’", "'").lower()
    return "".join((c for c in w if c.isalnum() or c in "-'"))


vocab_index = {normalise_word(w): i for i, w in enumerate(vocabulary)}
assert len(vocab_index) == 50


def filter_and_resample(x250):
    x = np.asarray(x250, dtype=np.float64)
    x = mne.filter.filter_data(x, 250.0, None, LOWPASS, n_jobs=-1, verbose=False)
    x = mne.filter.resample(x, up=FREQ, down=250.0, axis=-1, npad="auto", verbose=False)
    return x.T


def iqr_scale(x):
    q25, q75 = np.percentile(x, [25, 75], axis=0)
    scale = np.where(q75 - q25 > 0, q75 - q25, 1.0)
    return ((x / scale).astype(np.float16), scale.astype(np.float32))


def channel_names(attrs):
    names = attrs["channel_names"]
    if isinstance(names, bytes):
        names = names.decode()
    if isinstance(names, str):
        names = names.split(",")
    else:
        names = [
            name.decode() if isinstance(name, bytes) else str(name) for name in names
        ]
    return tuple(name.strip() for name in names)


def free_gb(p):
    return shutil.disk_usage(p).free / 2**30

In [ ]:
PAT = re.compile("sub-(\\d+)_ses-(\\d+)_task-Sherlock1_run-(\\d+)")
h5_of, tsv_of = ({}, {})
api = HfApi()
for repo in ("pnpl/LibriBrain2", "pnpl/LibriBrain"):
    for f in api.list_repo_tree(repo, repo_type="dataset", recursive=True, token=TOKEN):
        m = PAT.search(f.path)
        if not m or int(m.group(1)) not in LABELLED:
            continue
        key = (int(m.group(1)), m.group(2))
        if f.path.endswith(".h5"):
            assert key not in h5_of, f"two h5 files for {key}"
            desc = re.search("desc-([A-Za-z0-9]+)", f.path)
            h5_of[key] = (repo, f.path, desc.group(1) if desc else "full")
        elif f.path.endswith("_events.tsv"):
            tsv_of[key] = (repo, f.path)
EXPECT = {
    **{s: "full" for s in range(1, 13)},
    **{s: "firsthalf" for s in range(13, 23)},
    **{s: "firstquarter" for s in range(23, 33)},
}
jobs = [(s, ses) for s in LABELLED for ses in ("11", "12")]
for key in jobs:
    assert (
        key in h5_of and key in tsv_of
    ), f"missing h5 or events for sub-{key[0]} ses-{key[1]}"
    assert (
        h5_of[key][2] == EXPECT[key[0]]
    ), f"sub-{key[0]} ses-{key[1]} is {h5_of[key][2]}"
print(
    f"{len(jobs)} labelled recordings from {len(LABELLED)} subjects, each with its events file"
)

In [ ]:
t0 = time.time()
meg = open(CACHE_DIR / f"meg_{CACHE_VERSION}.bin", "wb")
word_ids = {w: i for i, w in enumerate(sorted(vocab_index))}
rows, runs, offset, names0 = ([], [], 0, None)
full_text = {}


def get_id(table, key):
    if key not in table:
        table[key] = len(table)
    return table[key]


for k, (s, ses) in enumerate(jobs, 1):
    assert (
        free_gb(DOWNLOAD_DIR) >= MIN_FREE_GB and free_gb(CACHE_DIR) >= MIN_FREE_GB
    ), "disk nearly full"
    repo, path, desc = h5_of[s, ses]
    h5 = hf_hub_download(
        repo, path, repo_type="dataset", token=TOKEN, local_dir=DOWNLOAD_DIR
    )
    with h5py.File(h5, "r") as f:
        attrs = dict(f.attrs)
        assert float(attrs["sample_frequency"]) == 250.0, attrs["sample_frequency"]
        assert float(attrs.get("highpass_cutoff", 0.1)) <= 0.1 + 1e-06
        assert float(attrs.get("lowpass_cutoff", 125.0)) >= LOWPASS
        names = channel_names(attrs)
        names0 = names0 or names
        assert names == names0, f"sub-{s} ses-{ses}: channel order differs"
        x, scale = iqr_scale(filter_and_resample(f["data"][:]))
    os.remove(h5)
    meg.write(x.tobytes())
    ev = pd.read_csv(
        hf_hub_download(
            *tsv_of[s, ses], repo_type="dataset", token=TOKEN, local_dir=DOWNLOAD_DIR
        ),
        sep="\t",
    )
    ev = ev[ev["kind"] == "word"].dropna(subset=["segment", "timemeg"])
    text = ev["segment"].astype(str).str.strip().map(normalise_word).to_numpy()
    sent = ev["sentenceidx"].fillna(-1).to_numpy(np.int64)
    widx = ev["wordidx"].fillna(-1).to_numpy(np.int64)
    full_text[s, ses] = (sent, widx, text)
    a = np.rint(ev["timemeg"].to_numpy(np.float64) * FREQ).astype(np.int64)
    keep = (a >= 0) & (a < len(x))
    run_id = len(runs)
    for i in np.flatnonzero(keep):
        rows.append(
            (offset + a[i], get_id(word_ids, text[i]), sent[i], widx[i], run_id)
        )
    runs.append((f"{s}/{ses}/Sherlock1/{desc}", s, ses, desc, offset, len(x), scale))
    offset += len(x)
    print(
        f"  [{k:2d}/{len(jobs)}] sub-{s:<2d} ses-{ses} {desc:12s} {len(x) / FREQ / 60:5.1f} min | {int(keep.sum()):4d} of {len(text)} words inside | total {offset / FREQ / 3600:5.2f} h | {(time.time() - t0) / 60:4.0f} min",
        flush=True,
    )
meg.close()
print(
    f"\n{offset / FREQ / 3600:.2f} h at {FREQ:g} Hz | {len(rows)} word events | {(time.time() - t0) / 60:.0f} min"
)

In [ ]:
import difflib

for ses in ("11", "12"):
    keys = sorted((k for k in full_text if k[1] == ses))
    ref_key = next((k for k in keys if k[0] <= 12))
    pairs = lambda k: list(zip(full_text[k][0].tolist(), full_text[k][2].tolist()))
    ref, same = (pairs(ref_key), [])
    print(
        f"ses-{ses}: reference sub-{ref_key[0]}, {len(ref)} words, {len(set((p[0] for p in ref)))} sentences"
    )
    for k in keys:
        t = pairs(k)
        if t == ref:
            same.append(k[0])
            continue
        sm = difflib.SequenceMatcher(None, ref, t, autojunk=False)
        matched = sum((b.size for b in sm.get_matching_blocks()))
        diffs = [op for op in sm.get_opcodes() if op[0] != "equal"]
        print(
            f"  sub-{k[0]}: {len(t)} words, {matched}/{len(ref)} matched, {len(diffs)} differences"
        )
        for tag, i1, i2, j1, j2 in diffs[:8]:
            print(f"    {tag:7s} reference {ref[i1:i2]} | this subject {t[j1:j2]}")
        assert matched >= 0.99 * len(
            ref
        ), f"sub-{k[0]} ses-{ses}: text differs too much to trust sentence ids"
    print(f"  identical to the reference: {len(same)} subjects")
r = np.array(rows, dtype=np.int64)
run_subject = np.array([x[1] for x in runs])
run_session = np.array([x[2] for x in runs])
print("\nlast sentence id with any word inside the recording:")
for grp, subs in (
    ("100%", range(1, 13)),
    ("50%", range(13, 23)),
    ("25%", range(23, 33)),
):
    for ses in ("11", "12"):
        ids = [
            np.flatnonzero((run_subject == s) & (run_session == ses))
            for s in subs
            if s in LABELLED
        ]
        last = [int(r[np.isin(r[:, 4], i), 2].max()) for i in ids if len(i)]
        if last:
            print(f"  {grp:>5s} ses-{ses}: {min(last)} to {max(last)}")

In [ ]:
hold = LibriBrainCompetitionHoldout(track="broad", data_path=str(HOLDOUT_DIR))
hmeta = pd.DataFrame(hold.metadata)
assert 0 not in set(hmeta["subject"]), "subject 0 belongs to the deep track"
assert (hmeta["index"].to_numpy() == np.arange(len(hmeta))).all()
print(hold.counts(), "| subjects", sorted(hmeta["subject"].unique()))
hold_bin = open(CACHE_DIR / f"hold_{CACHE_VERSION}.bin", "wb")
h_epoch, h_at, h_source, h_subject = (np.zeros(len(hmeta), np.int64) for _ in range(4))
e_start, e_len, h_scale, pos, n_epochs = ([], [], {}, 0, 0)
for s in sorted(hmeta["subject"].unique()):
    sentence_clips = np.load(hold._ensure_file(s, "sentence"))
    word_clips = np.load(hold._ensure_file(s, "word"))
    assert float(sentence_clips["sfreq"]) == float(word_clips["sfreq"]) == 250.0
    n_times = sentence_clips["sentence_n_times"].astype(np.int64)
    mask = np.asarray(sentence_clips["sentence_sample_mask"])
    for i, n in enumerate(n_times):
        assert mask[i, :n].all() and (
            not mask[i, n:].any()
        ), f"subj{s:02d} sentence {i}: mask is not a prefix"
    epochs = [
        filter_and_resample(sentence_clips["meg"][i, :, : n_times[i]])
        for i in range(len(n_times))
    ]
    n_sentence = len(epochs)
    epochs += [
        filter_and_resample(word_clips["meg"][i]) for i in range(len(word_clips["meg"]))
    ]
    flat, h_scale[s] = iqr_scale(np.concatenate(epochs, axis=0))
    assert np.isfinite(flat).all()
    hold_bin.write(flat.tobytes())
    lens = np.array([len(e) for e in epochs], dtype=np.int64)
    starts = pos + np.concatenate([[0], np.cumsum(lens)[:-1]])
    sel = np.flatnonzero(hmeta["subject"].to_numpy() == s)
    m = hmeta.iloc[sel]
    is_word = m["source"].to_numpy() == "word"
    local = np.where(
        is_word, n_sentence + m["epoch"].to_numpy(), m["epoch"].to_numpy()
    ).astype(np.int64)
    h_epoch[sel] = n_epochs + local
    h_at[sel] = starts[local] + np.rint(
        m["onset_s"].to_numpy(np.float64) * FREQ
    ).astype(np.int64)
    h_source[sel], h_subject[sel] = (is_word, s)
    e_start.append(starts)
    e_len.append(lens)
    pos, n_epochs = (pos + int(lens.sum()), n_epochs + len(epochs))
    for p in (hold._ensure_file(s, "sentence"), hold._ensure_file(s, "word")):
        os.remove(p)
    print(
        f"  subj{s:02d}: {n_sentence} + {len(epochs) - n_sentence} epochs, {len(sel)} rows",
        flush=True,
    )
hold_bin.close()
hold_start, hold_len = (np.concatenate(e_start), np.concatenate(e_len))
for e in np.unique(h_epoch):
    assert np.all(
        np.diff(h_at[h_epoch == e]) >= 0
    ), "rows within an epoch are not in onset order"
room = hold_start[h_epoch] + hold_len[h_epoch] - h_at
print(f"\n{n_epochs} epochs | {len(hmeta)} rows | {pos * 306 * 2 / 2 ** 20:.0f} MB")
for d in (1.0, 1.5, 2.0, 2.5):
    print(f"  {d:.1f} s servable for {(room >= d * FREQ).mean():6.1%} of rows")

In [ ]:
layout = mne.channels.read_layout("Vectorview-all")
key = {n.replace(" ", "").upper(): i for i, n in enumerate(layout.names)}
idx = [key[n.replace(" ", "").upper()] for n in names0]
positions = layout.pos[idx, :2].astype(np.float32)
positions = (positions - positions.min(0)) / (positions.max(0) - positions.min(0))
words = [w for w, _ in sorted(word_ids.items(), key=lambda kv: kv[1])]

if T5_CACHE_DIR is None:
    import torch
    from transformers import AutoTokenizer, T5EncoderModel

    tok = AutoTokenizer.from_pretrained(T5_MODEL)
    t5 = T5EncoderModel.from_pretrained(T5_MODEL).eval()
    emb = np.zeros((len(words), t5.config.d_model), dtype=np.float32)
    with torch.inference_mode():
        for a in range(0, len(words), 128):
            enc = tok(
                words[a : a + 128],
                return_tensors="pt",
                padding=True,
                add_special_tokens=False,
            )
            hs = t5(**enc, output_hidden_states=True).hidden_states
            m = enc["attention_mask"][..., None]
            emb[a : a + 128] = ((hs[T5_LAYER] * m).sum(1) / m.sum(1)).numpy()
    assert len(hs) == 25
    emb.astype(np.float16).tofile(CACHE_DIR / f"t5_{CACHE_VERSION}.bin")
    print(f"positions {positions.shape} | t5 {emb.shape} from layer {T5_LAYER}")
else:
    with np.load(
        T5_CACHE_DIR / f"meta_{CACHE_VERSION}.npz", allow_pickle=False
    ) as t5_metadata:
        assert np.array_equal(
            t5_metadata["words"], np.asarray(words)
        ), "T5 word order differs."
        assert np.array_equal(
            t5_metadata["vocab"], np.asarray(vocabulary)
        ), "T5 vocabulary differs."
        t5_dimension = int(t5_metadata["t5_dim"])
    source_t5 = T5_CACHE_DIR / f"t5_{CACHE_VERSION}.bin"
    assert (
        source_t5.stat().st_size == len(words) * t5_dimension * 2
    ), "Unexpected T5 file size."
    emb = np.memmap(source_t5, dtype=np.float16, mode="r").reshape(
        len(words), t5_dimension
    )
    shutil.copyfile(source_t5, CACHE_DIR / source_t5.name)
    print(f"T5 embeddings {emb.shape} loaded from {source_t5}")

In [ ]:
label = np.array([vocab_index.get(words[i], -1) for i in r[:, 1]], dtype=np.int16)
np.savez(
    CACHE_DIR / f"meta_{CACHE_VERSION}.npz",
    version=CACHE_VERSION,
    freq=FREQ,
    lowpass=LOWPASS,
    n_samples=offset,
    vocab=np.array(vocabulary),
    words=np.array(words),
    t5_dim=emb.shape[1],
    positions=positions,
    onset=r[:, 0],
    word_id=r[:, 1].astype(np.int32),
    label=label,
    sentence=r[:, 2],
    wordidx=r[:, 3],
    run=r[:, 4].astype(np.int32),
    run_name=np.array([x[0] for x in runs]),
    run_subject=run_subject,
    run_session=run_session,
    run_desc=np.array([x[3] for x in runs]),
    run_offset=np.array([x[4] for x in runs], dtype=np.int64),
    run_len=np.array([x[5] for x in runs], dtype=np.int64),
    run_scale=np.stack([x[6] for x in runs]),
    hold_subject=h_subject,
    hold_epoch=h_epoch,
    hold_at=h_at,
    hold_source=h_source.astype(np.int8),
    hold_start=hold_start,
    hold_len=hold_len,
    hold_indices=np.asarray(hold.indices),
    hold_scale=np.stack([h_scale[s] for s in sorted(h_scale)]),
)
per_run = np.bincount(r[label >= 0, 4], minlength=len(runs))
print(
    f"{len(r)} words, {(label >= 0).sum()} in the vocabulary ({(label >= 0).mean():.0%}), {len(words)} distinct words"
)
for grp, subs in (
    ("100%", range(1, 13)),
    ("50%", range(13, 23)),
    ("25%", range(23, 33)),
):
    n = [int(per_run[run_subject == s].sum()) for s in subs if s in LABELLED]
    if n:
        print(f"  {grp:>5s}: {min(n)} to {max(n)} scored words per subject")
missing = [vocabulary[c] for c in range(50) if not (label == c).any()]
print("vocabulary words absent from the broad training data:", missing or "none")
for f in sorted(CACHE_DIR.glob(f"*_{CACHE_VERSION}.*")):
    print(f"  {f.name:22s} {f.stat().st_size / 2 ** 30:6.2f} GB")
shutil.rmtree(DOWNLOAD_DIR, ignore_errors=True)
shutil.rmtree(HOLDOUT_DIR, ignore_errors=True)